# 05 — Close Baselines and Method Comparison

This notebook evaluates close calibration baselines against Source TS, Safe V2, and Frozen V3/SASC using the cached-logit workflow.

It supports the paper artifact for **Support-Aware Stream Calibration under Corruption Shift**.

**Notebook role**

- load ResNet and ViT cached-logit manifests;
- load source-bank, target-stream, and source-temperature tables;
- implement UTS-style weighted-NLL and SBTS-style close baselines;
- evaluate all close baselines against Source TS;
- compare close baselines with Safe V2 and Frozen V3/SASC;
- export compact comparison tables, bootstrap confidence intervals, and artifact manifests.

**Important note**

Target labels are not used for fitting the close baselines. Target labels are used only for downstream metric evaluation.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import shutil

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

**Paths and output folders**

In [ ]:
RESNET_ROOT = Path("/content/drive/MyDrive/AML_paper_freeze_v3")
VIT_ROOT = Path("/content/drive/MyDrive/AML_paper_freeze_v3_vit_extra/vit_analysis_same_pipeline")

RESNET_EXPORT_ROOT = RESNET_ROOT / "resnet_required_final_deliverables_for_supervisor"
VIT_EXPORT_ROOT = VIT_ROOT / "vit_required_final_deliverables_for_supervisor"

BASELINE_ROOT = Path("/content/drive/MyDrive/AML_close_baseline_clean")
BASELINE_TABLE_DIR = BASELINE_ROOT / "tables"
BASELINE_MANIFEST_DIR = BASELINE_ROOT / "manifests"

BASELINE_TABLE_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

print("ResNet root exists:", RESNET_ROOT.exists(), "|", RESNET_ROOT)
print("ViT root exists:", VIT_ROOT.exists(), "|", VIT_ROOT)
print("ResNet export exists:", RESNET_EXPORT_ROOT.exists(), "|", RESNET_EXPORT_ROOT)
print("ViT export exists:", VIT_EXPORT_ROOT.exists(), "|", VIT_EXPORT_ROOT)
print("Close-baseline output root:", BASELINE_ROOT)

**Protocol constants**

In [ ]:
BACKBONES = ["resnet18", "resnet50", "vit_b_16"]

SEEDS = [1, 2, 3]

CORRUPTIONS = [
    "brightness",
    "defocus_blur",
    "glass_blur",
    "motion_blur",
    "zoom_blur",
    "gaussian_noise",
    "shot_noise",
    "impulse_noise",
    "fog",
    "snow",
]

SEVERITIES = [1, 2, 3, 4, 5]

METRICS = [
    "accuracy",
    "mean_confidence",
    "signed_gap",
    "abs_signed_gap",
    "nll",
    "brier",
    "ece15",
    "aece15",
]

TEMPERATURE_GRID = np.linspace(0.50, 3.00, 251)

BASELINE_PROTOCOL = {
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "purpose": "Close-baseline feasibility and frozen-logit implementation.",
    "setting": {
        "cached_logits_available": True,
        "source_bank_logits_available": True,
        "target_labels_used_for_fitting": False,
        "target_labels_used_for_evaluation": True,
        "no_model_retraining": True,
        "no_new_forward_pass_preferred": True,
    },
    "implemented_baselines": [
        "uts_weighted_nll_unconstrained",
        "uts_weighted_nll_no_sharpen",
        "sbts_entropy_shift_ridge",
        "sbts_entropy_shift_ridge_no_sharpen",
        "sbts_severity_ridge",
        "sbts_severity_ridge_no_sharpen",
    ],
    "not_implemented_now": [
        "PseudoCal",
        "UTDC",
    ],
    "temperature_grid": {
        "min": float(TEMPERATURE_GRID.min()),
        "max": float(TEMPERATURE_GRID.max()),
        "n_grid": int(len(TEMPERATURE_GRID)),
    },
    "important_note": (
        "UTS is implemented as a frozen-logit UTS-style weighted-NLL baseline. "
        "SBTS is implemented as a frozen-logit shift-intensity mapping using source-bank fitted temperatures. "
        "PseudoCal is not implemented because a faithful version likely requires target image/feature access "
        "and new forward passes. UTDC is not implemented until the exact method is verified."
    ),
}

protocol_path = BASELINE_MANIFEST_DIR / "close_baseline_protocol.json"

with open(protocol_path, "w") as f:
    json.dump(BASELINE_PROTOCOL, f, indent=2)

print("Saved protocol:", protocol_path)
print(json.dumps(BASELINE_PROTOCOL, indent=2))

**Feasibility table**

In [ ]:
feasibility_rows = [
    {
        "baseline": "UTS",
        "baseline_label_for_our_work": "UTS-style weighted NLL",
        "main_idea": (
            "Fit temperature using unlabeled target/test logits through an unsupervised weighted-NLL objective."
        ),
        "required_inputs": "Cached unlabeled target logits.",
        "needs_target_labels_for_fitting": False,
        "needs_target_labels_for_evaluation": True,
        "needs_images": False,
        "needs_features": False,
        "needs_model_forward_pass": False,
        "works_with_cached_logits": True,
        "works_with_frozen_logit_stream_setting": True,
        "implementable_now": True,
        "implementation_status": "Implemented as UTS-style weighted NLL.",
        "fairness_note": (
            "We should label it UTS-style unless we exactly reproduce the original paper's weighting rule."
        ),
        "priority": 1,
    },
    {
        "baseline": "SBTS-style",
        "baseline_label_for_our_work": "SBTS-style shift-intensity temperature mapping",
        "main_idea": (
            "Use source-bank pseudo-domains to learn a mapping from shift intensity or stream descriptors "
            "to fitted temperature."
        ),
        "required_inputs": (
            "Source-bank descriptors, source-bank fitted temperatures, and unlabeled target stream descriptors. "
            "Severity-based variant also uses benchmark severity metadata."
        ),
        "needs_target_labels_for_fitting": False,
        "needs_target_labels_for_evaluation": True,
        "needs_images": False,
        "needs_features": False,
        "needs_model_forward_pass": False,
        "works_with_cached_logits": True,
        "works_with_frozen_logit_stream_setting": True,
        "implementable_now": True,
        "implementation_status": "Implemented as entropy-shift ridge and severity ridge variants.",
        "fairness_note": (
            "Call it SBTS-style unless we exactly reproduce a named SBTS method. "
            "Severity version is benchmark-metadata dependent; entropy-shift version is more deployable."
        ),
        "priority": 2,
    },
    {
        "baseline": "PseudoCal",
        "baseline_label_for_our_work": "PseudoCal feasibility only",
        "main_idea": (
            "Synthesize a labeled pseudo-target calibration set from unlabeled target data, then fit temperature scaling."
        ),
        "required_inputs": (
            "Usually target images/features and inference-stage mixup or new model forward passes."
        ),
        "needs_target_labels_for_fitting": False,
        "needs_target_labels_for_evaluation": True,
        "needs_images": True,
        "needs_features": "Possibly",
        "needs_model_forward_pass": True,
        "works_with_cached_logits": "Not faithfully",
        "works_with_frozen_logit_stream_setting": False,
        "implementable_now": False,
        "implementation_status": "Not implemented.",
        "fairness_note": (
            "A faithful implementation goes beyond our frozen-logit stream setting."
        ),
        "priority": 3,
    },
    {
        "baseline": "UTDC",
        "baseline_label_for_our_work": "UTDC feasibility pending",
        "main_idea": "Exact method must be verified before implementation.",
        "required_inputs": "Unknown until exact paper/method is confirmed.",
        "needs_target_labels_for_fitting": "Unknown",
        "needs_target_labels_for_evaluation": True,
        "needs_images": "Unknown",
        "needs_features": "Unknown",
        "needs_model_forward_pass": "Unknown",
        "works_with_cached_logits": "Unknown",
        "works_with_frozen_logit_stream_setting": "Unknown",
        "implementable_now": False,
        "implementation_status": "Not implemented.",
        "fairness_note": (
            "Do not claim implementability until the exact UTDC method is identified."
        ),
        "priority": 4,
    },
]

feasibility_df = pd.DataFrame(feasibility_rows)

feasibility_path = BASELINE_TABLE_DIR / "close_baseline_feasibility_table.csv"
feasibility_df.to_csv(feasibility_path, index=False)

compact_cols = [
    "baseline",
    "baseline_label_for_our_work",
    "works_with_cached_logits",
    "works_with_frozen_logit_stream_setting",
    "needs_target_labels_for_fitting",
    "needs_images",
    "needs_features",
    "needs_model_forward_pass",
    "implementable_now",
    "implementation_status",
    "priority",
    "fairness_note",
]

feasibility_compact_df = feasibility_df[compact_cols].copy()

feasibility_compact_path = BASELINE_TABLE_DIR / "close_baseline_feasibility_compact.csv"
feasibility_compact_df.to_csv(feasibility_compact_path, index=False)

print("Saved full feasibility table:", feasibility_path)
print("Saved compact feasibility table:", feasibility_compact_path)

display(feasibility_df)

**Basic loading helpers**

In [ ]:
def normalize_backbone_column(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "architecture" in df.columns and "backbone" not in df.columns:
        df = df.rename(columns={"architecture": "backbone"})

    return df


def load_existing_csv(path: Path, required: bool = True) -> pd.DataFrame:
    path = Path(path)

    if not path.exists():
        if required:
            raise FileNotFoundError(f"Missing required CSV: {path}")
        return None

    df = pd.read_csv(path)
    df = normalize_backbone_column(df)

    return df


def find_first_existing(candidates, label: str) -> Path:
    for p in candidates:
        p = Path(p)
        if p.exists():
            print(f"Using {label}:", p)
            return p

    print(f"\nCould not find {label}. Candidate paths checked:")
    for p in candidates:
        print("-", p)

    raise FileNotFoundError(f"Could not find {label}.")

**Logit loading helpers**

In [ ]:
def load_logits_labels_any(path: Path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Missing logits file: {path}")

    if path.suffix == ".pt":
        blob = torch.load(path, map_location="cpu", weights_only=False)

        if "logits" not in blob:
            raise KeyError(f"No 'logits' key in {path}. Keys: {list(blob.keys())}")

        if "labels" not in blob:
            raise KeyError(f"No 'labels' key in {path}. Keys: {list(blob.keys())}")

        logits = blob["logits"].detach().cpu().float()
        labels = blob["labels"].detach().cpu().long()

        return logits, labels

    if path.suffix == ".npz":
        blob = np.load(path, allow_pickle=True)

        if "logits" in blob.files:
            logits_np = blob["logits"]
        elif "arr_0" in blob.files:
            logits_np = blob["arr_0"]
        else:
            raise KeyError(f"Could not find logits key in {path}. Keys: {blob.files}")

        if "labels" in blob.files:
            labels_np = blob["labels"]
        elif "y" in blob.files:
            labels_np = blob["y"]
        elif "arr_1" in blob.files:
            labels_np = blob["arr_1"]
        else:
            raise KeyError(f"Could not find labels key in {path}. Keys: {blob.files}")

        logits = torch.tensor(logits_np, dtype=torch.float32)
        labels = torch.tensor(labels_np, dtype=torch.long)

        return logits, labels

    raise ValueError(f"Unsupported logits file type: {path}")

**Metric helpers**

In [ ]:
def apply_temperature(logits: torch.Tensor, temperature: float) -> torch.Tensor:
    temperature = float(temperature)
    assert temperature > 0, f"Temperature must be positive, got {temperature}"
    return logits.detach().cpu().float() / temperature


def softmax_probs(logits: torch.Tensor) -> torch.Tensor:
    return torch.softmax(logits.float(), dim=1)


def brier_from_probs(probs: torch.Tensor, labels: torch.Tensor) -> float:
    one_hot = F.one_hot(labels.long(), num_classes=probs.shape[1]).float()
    return float(((probs - one_hot) ** 2).sum(dim=1).mean().item())


def ece_fixed_bins(probs: torch.Tensor, labels: torch.Tensor, n_bins: int = 15) -> float:
    confidence, prediction = probs.max(dim=1)
    correct = prediction.eq(labels.long()).float()

    bin_edges = torch.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        lo = bin_edges[i]
        hi = bin_edges[i + 1]

        if i == 0:
            in_bin = (confidence >= lo) & (confidence <= hi)
        else:
            in_bin = (confidence > lo) & (confidence <= hi)

        prop = in_bin.float().mean().item()

        if prop > 0:
            acc_bin = correct[in_bin].mean().item()
            conf_bin = confidence[in_bin].mean().item()
            ece += abs(acc_bin - conf_bin) * prop

    return float(ece)


def adaptive_ece(probs: torch.Tensor, labels: torch.Tensor, n_bins: int = 15) -> float:
    confidence, prediction = probs.max(dim=1)
    correct = prediction.eq(labels.long()).float()

    confidence = confidence.detach().cpu()
    correct = correct.detach().cpu()

    n = len(confidence)

    if n == 0:
        return 0.0

    order = torch.argsort(confidence)
    confidence = confidence[order]
    correct = correct[order]

    bin_sizes = [n // n_bins] * n_bins

    for i in range(n % n_bins):
        bin_sizes[i] += 1

    aece = 0.0
    start = 0

    for size in bin_sizes:
        if size == 0:
            continue

        end = start + size
        conf_bin = confidence[start:end]
        acc_bin = correct[start:end]

        aece += abs(conf_bin.mean().item() - acc_bin.mean().item()) * (size / n)
        start = end

    return float(aece)


def metrics_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> dict:
    logits = logits.detach().cpu().float()
    labels = labels.detach().cpu().long()

    probs = softmax_probs(logits)
    confidence, prediction = probs.max(dim=1)
    correct = prediction.eq(labels).float()

    accuracy = float(correct.mean().item())
    mean_confidence = float(confidence.mean().item())
    signed_gap = mean_confidence - accuracy

    out = {
        "accuracy": accuracy,
        "mean_confidence": mean_confidence,
        "signed_gap": float(signed_gap),
        "abs_signed_gap": float(abs(signed_gap)),
        "nll": float(F.cross_entropy(logits, labels, reduction="mean").item()),
        "brier": brier_from_probs(probs, labels),
        "ece15": ece_fixed_bins(probs, labels, n_bins=15),
        "aece15": adaptive_ece(probs, labels, n_bins=15),
    }

    assert 0.0 <= out["ece15"] <= 1.0, f"Invalid ECE: {out['ece15']}"
    assert 0.0 <= out["aece15"] <= 1.0, f"Invalid AECE: {out['aece15']}"

    return out

**Standardize target logits manifests**

In [ ]:
def standardize_logits_manifest(
    df: pd.DataFrame,
    root: Path,
    default_backbone: str = None,
) -> pd.DataFrame:
    """
    Standardize target logits manifest to:
    backbone, seed, corruption, severity, condition, path
    """
    df = normalize_backbone_column(df.copy())

    path_candidates = [
        "path",
        "logits_path",
        "file_path",
        "cache_path",
        "cached_path",
        "target_logits_path",
        "logit_path",
    ]

    found_path_col = None

    for c in path_candidates:
        if c in df.columns:
            found_path_col = c
            break

    if found_path_col is None:
        path_like_cols = [c for c in df.columns if "path" in c.lower()]
        if len(path_like_cols) > 0:
            found_path_col = path_like_cols[0]

    assert found_path_col is not None, (
        "No usable path column found in manifest. "
        f"Columns are: {list(df.columns)}"
    )

    if found_path_col != "path":
        df = df.rename(columns={found_path_col: "path"})

    if "backbone" not in df.columns:
        assert default_backbone is not None, (
            "backbone column missing and no default_backbone was provided."
        )
        df["backbone"] = default_backbone

    if "corruption_name" in df.columns and "corruption" not in df.columns:
        df = df.rename(columns={"corruption_name": "corruption"})

    if "severity_level" in df.columns and "severity" not in df.columns:
        df = df.rename(columns={"severity_level": "severity"})

    if "seed_id" in df.columns and "seed" not in df.columns:
        df = df.rename(columns={"seed_id": "seed"})

    if "condition" not in df.columns:
        assert "corruption" in df.columns and "severity" in df.columns, (
            "Cannot recover condition because corruption/severity columns are missing. "
            f"Columns are: {list(df.columns)}"
        )
        df["condition"] = (
            df["corruption"].astype(str)
            + "_sev"
            + df["severity"].astype(str)
        )

    required_cols = [
        "backbone",
        "seed",
        "corruption",
        "severity",
        "condition",
        "path",
    ]

    missing = [c for c in required_cols if c not in df.columns]
    assert len(missing) == 0, (
        f"Manifest missing required columns: {missing}. "
        f"Columns are: {list(df.columns)}"
    )

    df = df[required_cols].copy()

    df["backbone"] = df["backbone"].astype(str)
    df["seed"] = df["seed"].astype(int)
    df["corruption"] = df["corruption"].astype(str)
    df["severity"] = df["severity"].astype(int)
    df["condition"] = df["condition"].astype(str)

    def resolve_path(x):
        p = Path(str(x))

        if p.exists():
            return str(p)

        if not p.is_absolute():
            candidate = root / p
            if candidate.exists():
                return str(candidate)

        return str(p)

    df["path"] = df["path"].apply(resolve_path)

    df = df[
        df["backbone"].isin(BACKBONES)
        & df["corruption"].isin(CORRUPTIONS)
        & df["severity"].isin(SEVERITIES)
    ].copy()

    df = df.drop_duplicates(
        subset=["backbone", "seed", "corruption", "severity", "condition"]
    ).reset_index(drop=True)

    return df

**Load target logits manifests**

In [ ]:
# ResNet target logits manifest
resnet_target_manifest_path = RESNET_ROOT / "manifests" / "cached_logits_manifest.csv"

resnet_target_manifest_raw = load_existing_csv(
    resnet_target_manifest_path,
    required=True,
)

resnet_target_manifest = standardize_logits_manifest(
    df=resnet_target_manifest_raw,
    root=RESNET_ROOT,
    default_backbone=None,
)

print("ResNet target manifest rows:", len(resnet_target_manifest))
display(resnet_target_manifest.head())


# ViT target logits manifest
vit_manifest_dir = VIT_ROOT / "manifests"

vit_manifest_candidates = [
    vit_manifest_dir / "vit_cached_target_logits_manifest.csv",
    vit_manifest_dir / "cached_logits_manifest.csv",
    vit_manifest_dir / "target_logits_manifest.csv",
    vit_manifest_dir / "imagenetc_logits_manifest.csv",
]

vit_target_manifest_path = find_first_existing(
    vit_manifest_candidates,
    label="ViT target logits manifest",
)

vit_target_manifest_raw = load_existing_csv(
    vit_target_manifest_path,
    required=True,
)

vit_target_manifest = standardize_logits_manifest(
    df=vit_target_manifest_raw,
    root=VIT_ROOT,
    default_backbone="vit_b_16",
)

print("ViT target manifest rows:", len(vit_target_manifest))
display(vit_target_manifest.head())


target_manifest = pd.concat(
    [resnet_target_manifest, vit_target_manifest],
    ignore_index=True,
    sort=False,
)

target_manifest = target_manifest[
    target_manifest["backbone"].isin(BACKBONES)
].copy()

target_manifest = target_manifest.drop_duplicates(
    subset=["backbone", "seed", "corruption", "severity", "condition"]
).reset_index(drop=True)

expected_target_rows = len(BACKBONES) * len(SEEDS) * len(CORRUPTIONS) * len(SEVERITIES)

print("Expected target rows:", expected_target_rows)
print("Actual target rows:", len(target_manifest))

assert len(target_manifest) == expected_target_rows, (
    f"Expected {expected_target_rows} target logit rows, got {len(target_manifest)}"
)

missing_logit_files = [
    p for p in target_manifest["path"].tolist()
    if not Path(p).exists()
]

print("Missing target logit files:", len(missing_logit_files))

assert len(missing_logit_files) == 0, (
    "Missing target logit files. First examples:\n"
    + "\n".join(missing_logit_files[:10])
)

target_manifest_path = BASELINE_MANIFEST_DIR / "combined_target_logits_manifest.csv"
target_manifest.to_csv(target_manifest_path, index=False)

print("Saved combined target manifest:", target_manifest_path)
display(target_manifest.head())

**Load source-bank descriptor tables**

In [ ]:
resnet_source_bank_path = (
    RESNET_ROOT
    / "stream_methods"
    / "regenerated_source_bank_stream_descriptor_table.csv"
)

vit_source_bank_path = (
    VIT_ROOT
    / "stream_methods"
    / "regenerated_source_bank_stream_descriptor_table.csv"
)

resnet_source_bank = load_existing_csv(resnet_source_bank_path, required=True)
vit_source_bank = load_existing_csv(vit_source_bank_path, required=True)

source_bank_df = pd.concat(
    [resnet_source_bank, vit_source_bank],
    ignore_index=True,
    sort=False,
)

source_bank_df = normalize_backbone_column(source_bank_df)

source_bank_df = source_bank_df[
    source_bank_df["backbone"].isin(BACKBONES)
].copy()

source_bank_df["seed"] = source_bank_df["seed"].astype(int)
source_bank_df["severity"] = source_bank_df["severity"].astype(int)
source_bank_df["corruption"] = source_bank_df["corruption"].astype(str)

required_source_bank_cols = [
    "backbone",
    "seed",
    "corruption",
    "severity",
    "condition",
    "entropy_mean",
    "source_bank_target_temperature",
    "source_bank_target_log_temperature",
]

missing_cols = [c for c in required_source_bank_cols if c not in source_bank_df.columns]

assert len(missing_cols) == 0, (
    f"Missing source-bank columns: {missing_cols}"
)

expected_source_bank_rows = len(BACKBONES) * len(SEEDS) * len(CORRUPTIONS) * len(SEVERITIES)

print("Expected source-bank rows:", expected_source_bank_rows)
print("Actual source-bank rows:", len(source_bank_df))

assert len(source_bank_df) == expected_source_bank_rows, (
    f"Expected {expected_source_bank_rows}, got {len(source_bank_df)}"
)

source_bank_combined_path = BASELINE_TABLE_DIR / "combined_source_bank_descriptor_table.csv"
source_bank_df.to_csv(source_bank_combined_path, index=False)

print("Saved combined source-bank descriptor table:", source_bank_combined_path)
display(source_bank_df.head())

**Load target stream descriptor tables**

In [ ]:
resnet_target_stream_path = (
    RESNET_ROOT
    / "stream_methods"
    / "regenerated_target_stream_descriptor_table.csv"
)

vit_target_stream_path = (
    VIT_ROOT
    / "stream_methods"
    / "regenerated_target_stream_descriptor_table.csv"
)

resnet_target_stream = load_existing_csv(resnet_target_stream_path, required=True)
vit_target_stream = load_existing_csv(vit_target_stream_path, required=True)

target_stream_df = pd.concat(
    [resnet_target_stream, vit_target_stream],
    ignore_index=True,
    sort=False,
)

target_stream_df = normalize_backbone_column(target_stream_df)

target_stream_df = target_stream_df[
    target_stream_df["backbone"].isin(BACKBONES)
].copy()

target_stream_df["seed"] = target_stream_df["seed"].astype(int)
target_stream_df["severity"] = target_stream_df["severity"].astype(int)
target_stream_df["corruption"] = target_stream_df["corruption"].astype(str)

required_target_stream_cols = [
    "backbone",
    "seed",
    "corruption",
    "severity",
    "condition",
    "entropy_mean",
]

missing_cols = [c for c in required_target_stream_cols if c not in target_stream_df.columns]

assert len(missing_cols) == 0, (
    f"Missing target stream descriptor columns: {missing_cols}"
)

expected_target_stream_rows = len(BACKBONES) * len(SEEDS) * len(CORRUPTIONS) * len(SEVERITIES)

print("Expected target stream rows:", expected_target_stream_rows)
print("Actual target stream rows:", len(target_stream_df))

assert len(target_stream_df) == expected_target_stream_rows, (
    f"Expected {expected_target_stream_rows}, got {len(target_stream_df)}"
)

target_stream_combined_path = BASELINE_TABLE_DIR / "combined_target_stream_descriptor_table.csv"
target_stream_df.to_csv(target_stream_combined_path, index=False)

print("Saved combined target stream descriptor table:", target_stream_combined_path)
display(target_stream_df.head())

**Load source TS tables**

In [ ]:
resnet_source_ts_path = (
    RESNET_ROOT
    / "source_ts"
    / "refit_source_ts_from_clean_source_logits.csv"
)

vit_source_ts_path = find_first_existing(
    [
        VIT_ROOT / "source_ts" / "refit_source_ts_from_clean_source_logits.csv",
        VIT_ROOT / "source_ts" / "stage3_vit_refit_source_ts_from_calib_logits.csv",
    ],
    label="ViT source TS table",
)

resnet_source_ts = load_existing_csv(resnet_source_ts_path, required=True)
vit_source_ts = load_existing_csv(vit_source_ts_path, required=True)

source_ts_df = pd.concat(
    [resnet_source_ts, vit_source_ts],
    ignore_index=True,
    sort=False,
)

source_ts_df = normalize_backbone_column(source_ts_df)

if "temperature" not in source_ts_df.columns and "source_T" in source_ts_df.columns:
    source_ts_df = source_ts_df.rename(columns={"source_T": "temperature"})

required_source_ts_cols = [
    "backbone",
    "seed",
    "temperature",
]

missing_cols = [c for c in required_source_ts_cols if c not in source_ts_df.columns]

assert len(missing_cols) == 0, (
    f"Missing source TS columns: {missing_cols}. "
    f"Available columns: {list(source_ts_df.columns)}"
)

source_ts_df = source_ts_df[
    source_ts_df["backbone"].isin(BACKBONES)
].copy()

source_ts_df["seed"] = source_ts_df["seed"].astype(int)
source_ts_df["temperature"] = source_ts_df["temperature"].astype(float)

expected_source_ts_rows = len(BACKBONES) * len(SEEDS)

print("Expected source TS rows:", expected_source_ts_rows)
print("Actual source TS rows:", len(source_ts_df))

assert len(source_ts_df) == expected_source_ts_rows, (
    f"Expected {expected_source_ts_rows}, got {len(source_ts_df)}"
)

SOURCE_T = {
    (row["backbone"], int(row["seed"])): float(row["temperature"])
    for _, row in source_ts_df.iterrows()
}

source_ts_combined_path = BASELINE_TABLE_DIR / "combined_source_ts_table.csv"
source_ts_df.to_csv(source_ts_combined_path, index=False)

print("Saved combined source TS table:", source_ts_combined_path)
display(source_ts_df)

**Build source entropy reference table**

In [ ]:
def find_source_calibration_path_column(df: pd.DataFrame):
    candidate_cols = [
        "source_calibration_logits_path",
        "calibration_logits_path",
        "calib_logits_path",
        "path",
        "logits_path",
    ]

    for c in candidate_cols:
        if c in df.columns:
            return c

    path_like_cols = [c for c in df.columns if "path" in c.lower()]

    if len(path_like_cols) > 0:
        return path_like_cols[0]

    return None


def predictive_entropy_from_logits(
    logits: torch.Tensor,
    temperature: float = 1.0,
) -> float:
    scaled_logits = apply_temperature(logits, temperature)
    probs = softmax_probs(scaled_logits)
    entropy = -(probs * torch.log(probs.clamp_min(1e-12))).sum(dim=1)
    return float(entropy.mean().item())


def mean_confidence_from_logits(
    logits: torch.Tensor,
    temperature: float = 1.0,
) -> float:
    scaled_logits = apply_temperature(logits, temperature)
    probs = softmax_probs(scaled_logits)
    confidence = probs.max(dim=1).values
    return float(confidence.mean().item())


source_entropy_rows = []

path_col = find_source_calibration_path_column(source_ts_df)

print("Detected source calibration path column:", path_col)

for _, row in source_ts_df.iterrows():
    backbone = row["backbone"]
    seed = int(row["seed"])
    source_T = float(row["temperature"])

    entropy_ref = np.nan
    confidence_ref = np.nan
    reference_source = "not_available"

    if path_col is not None and pd.notna(row.get(path_col, np.nan)):
        candidate_path = Path(str(row[path_col]))

        if candidate_path.exists():
            try:
                logits, labels = load_logits_labels_any(candidate_path)

                entropy_ref = predictive_entropy_from_logits(
                    logits,
                    temperature=source_T,
                )

                confidence_ref = mean_confidence_from_logits(
                    logits,
                    temperature=source_T,
                )

                reference_source = str(candidate_path)

            except Exception as e:
                print(
                    "Could not load source calibration logits:",
                    candidate_path,
                    "|",
                    repr(e),
                )

    if not np.isfinite(entropy_ref):
        source_bank_subset = source_bank_df[
            (source_bank_df["backbone"] == backbone)
            & (source_bank_df["seed"] == seed)
        ].copy()

        assert len(source_bank_subset) == 50, (
            f"{backbone} seed={seed}: expected 50 source-bank rows, "
            f"got {len(source_bank_subset)}"
        )

        entropy_ref = float(source_bank_subset["entropy_mean"].median())
        confidence_ref = np.nan
        reference_source = "fallback_median_source_bank_entropy"

    source_entropy_rows.append({
        "backbone": backbone,
        "seed": seed,
        "source_T": source_T,
        "source_entropy_ref": float(entropy_ref),
        "source_confidence_ref": (
            float(confidence_ref)
            if np.isfinite(confidence_ref)
            else np.nan
        ),
        "reference_source": reference_source,
    })

source_entropy_ref_df = pd.DataFrame(source_entropy_rows)

expected_rows = len(BACKBONES) * len(SEEDS)

assert len(source_entropy_ref_df) == expected_rows
assert source_entropy_ref_df["source_entropy_ref"].notna().all()

SOURCE_ENTROPY_REF = {
    (row["backbone"], int(row["seed"])): float(row["source_entropy_ref"])
    for _, row in source_entropy_ref_df.iterrows()
}

source_entropy_ref_path = BASELINE_TABLE_DIR / "source_entropy_reference_table.csv"
source_entropy_ref_df.to_csv(source_entropy_ref_path, index=False)

print("Saved source entropy reference table:", source_entropy_ref_path)
display(source_entropy_ref_df)

**UTS-style weighted NLL baseline**

In [ ]:
def select_temperature_uts_weighted_nll(
    logits: torch.Tensor,
    source_T: float,
    no_sharpen: bool = False,
    temperature_grid=TEMPERATURE_GRID,
    weight_mode: str = "one_minus_confidence",
    eps: float = 1e-12,
):
    """
    Frozen-logit UTS-style weighted NLL.

    Uses only unlabeled target logits for fitting.

    Pseudo-label:
        y_hat_i = argmax softmax(logits_i)

    Objective:
        mean_i w_i * CE(logits_i / T, y_hat_i)

    Target labels are NOT used.
    """

    logits = logits.detach().cpu().float()
    source_T = float(source_T)

    with torch.no_grad():
        probs_raw = torch.softmax(logits, dim=1)
        confidence_raw, pseudo_labels = probs_raw.max(dim=1)

        if weight_mode == "one_minus_confidence":
            weights = 1.0 - confidence_raw

        elif weight_mode == "entropy":
            entropy = -(probs_raw * torch.log(probs_raw.clamp_min(eps))).sum(dim=1)
            weights = entropy / np.log(logits.shape[1])

        elif weight_mode == "margin":
            sorted_probs, _ = torch.sort(probs_raw, dim=1, descending=True)
            margin = sorted_probs[:, 0] - sorted_probs[:, 1]
            weights = 1.0 - margin

        elif weight_mode == "uniform":
            weights = torch.ones_like(confidence_raw)

        else:
            raise ValueError(f"Unknown weight_mode: {weight_mode}")

        weights = weights.detach().cpu().float()
        weights = weights + eps
        weights = weights / weights.mean().clamp_min(eps)

    candidate_T = np.array(temperature_grid, dtype=float)

    if no_sharpen:
        candidate_T = candidate_T[candidate_T >= source_T - 1e-12]

    if len(candidate_T) == 0:
        candidate_T = np.array([source_T], dtype=float)

    best_T = None
    best_loss = float("inf")
    best_mean_confidence = None
    best_mean_entropy = None

    for T in candidate_T:
        scaled_logits = logits / float(T)

        per_sample_ce = F.cross_entropy(
            scaled_logits,
            pseudo_labels.long(),
            reduction="none",
        )

        loss = float((weights * per_sample_ce).mean().item())

        if loss < best_loss:
            probs_scaled = torch.softmax(scaled_logits, dim=1)
            conf_scaled = probs_scaled.max(dim=1).values
            entropy_scaled = -(
                probs_scaled * torch.log(probs_scaled.clamp_min(eps))
            ).sum(dim=1)

            best_loss = loss
            best_T = float(T)
            best_mean_confidence = float(conf_scaled.mean().item())
            best_mean_entropy = float(entropy_scaled.mean().item())

    return {
        "predicted_T": float(best_T),
        "raw_predicted_T": float(best_T),
        "objective_value": float(best_loss),
        "weight_mode": weight_mode,
        "pseudo_label_source": "argmax_raw_logits",
        "mean_confidence_after_scaling": float(best_mean_confidence),
        "mean_entropy_after_scaling": float(best_mean_entropy),
        "no_sharpen": bool(no_sharpen),
        "target_labels_used_for_fit": False,
    }


print("UTS-style weighted NLL helper ready.")

**SBTS-style fitting and prediction helpers**

In [ ]:
def fit_sbts_models(
    train_df: pd.DataFrame,
    source_T: float,
):
    """
    SBTS-style frozen-logit shift-to-temperature mappings.

    Source-bank conditions act as pseudo-domains:
        descriptor/shift intensity -> fitted source-bank log-temperature

    Target labels are NOT used.
    """

    train_df = train_df.copy()
    source_T = float(source_T)

    required_cols = [
        "severity",
        "entropy_mean",
        "source_bank_target_temperature",
        "source_bank_target_log_temperature",
    ]

    missing = [c for c in required_cols if c not in train_df.columns]
    assert len(missing) == 0, f"Missing SBTS columns: {missing}"

    y_logT = train_df["source_bank_target_log_temperature"].astype(float).to_numpy()

    # 1. Severity-only mapping

    X_severity = train_df[["severity"]].astype(float).to_numpy()

    severity_model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0),
    )

    severity_model.fit(X_severity, y_logT)


    # 2. Entropy-shift mapping
    entropy_train_median = float(train_df["entropy_mean"].median())

    train_df["entropy_shift_from_source_bank_median"] = (
        train_df["entropy_mean"].astype(float) - entropy_train_median
    )

    X_entropy_shift = (
        train_df[["entropy_shift_from_source_bank_median"]]
        .astype(float)
        .to_numpy()
    )

    entropy_shift_model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0),
    )

    entropy_shift_model.fit(X_entropy_shift, y_logT)

    # Conservative empirical clipping range from source-bank fitted temperatures.
    T_min_empirical = max(
        0.05,
        float(train_df["source_bank_target_temperature"].quantile(0.02)),
    )

    T_max_empirical = min(
        10.0,
        float(train_df["source_bank_target_temperature"].quantile(0.98)),
    )

    return {
        "severity_model": severity_model,
        "entropy_shift_model": entropy_shift_model,
        "entropy_train_median": entropy_train_median,
        "T_min_empirical": float(T_min_empirical),
        "T_max_empirical": float(T_max_empirical),
        "source_T": float(source_T),
        "n_train_conditions": int(len(train_df)),
    }


def predict_sbts_temperature(
    model_info: dict,
    row: pd.Series,
    variant: str,
    no_sharpen: bool = False,
):
    """
    Predict temperature using an SBTS-style mapping.

    Supported variants:
        sbts_severity_ridge
        sbts_entropy_shift_ridge
    """

    source_T = float(model_info["source_T"])

    if variant == "sbts_severity_ridge":
        X = np.array([[float(row["severity"])]], dtype=float)
        raw_logT = float(model_info["severity_model"].predict(X)[0])

    elif variant == "sbts_entropy_shift_ridge":
        entropy_shift = (
            float(row["entropy_mean"])
            - float(model_info["entropy_train_median"])
        )

        X = np.array([[entropy_shift]], dtype=float)
        raw_logT = float(model_info["entropy_shift_model"].predict(X)[0])

    else:
        raise ValueError(f"Unknown SBTS variant: {variant}")

    raw_T = float(np.exp(raw_logT))

    T_min = float(model_info["T_min_empirical"])
    T_max = float(model_info["T_max_empirical"])

    if no_sharpen:
        T_min = max(T_min, source_T)

    pred_T = float(np.clip(raw_T, T_min, T_max))

    return {
        "predicted_T": pred_T,
        "raw_predicted_T": raw_T,
        "raw_logT": raw_logT,
        "T_min": T_min,
        "T_max": T_max,
        "no_sharpen": bool(no_sharpen),
        "target_labels_used_for_fit": False,
    }


print("SBTS-style helpers ready.")

**Prepare target lookup and sanity checks**

In [ ]:
target_lookup = target_manifest.set_index(
    ["backbone", "seed", "corruption", "severity", "condition"]
)

duplicate_target_keys = target_manifest.duplicated(
    subset=["backbone", "seed", "corruption", "severity", "condition"]
).sum()

assert duplicate_target_keys == 0, f"Duplicate target manifest keys: {duplicate_target_keys}"


def get_logits_path_from_lookup(
    lookup: pd.DataFrame,
    key: tuple,
) -> Path:
    row = lookup.loc[key]

    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]

    return Path(str(row["path"]))


required_prediction_keys = []

for backbone in BACKBONES:
    for seed in SEEDS:
        for corruption in CORRUPTIONS:
            for severity in SEVERITIES:
                condition = f"{corruption}_sev{severity}"
                required_prediction_keys.append(
                    (backbone, seed, corruption, severity, condition)
                )

missing_keys = [
    key for key in required_prediction_keys
    if key not in target_lookup.index
]

print("Missing target lookup keys:", len(missing_keys))

assert len(missing_keys) == 0, (
    "Missing target lookup keys. First examples:\n"
    + "\n".join(map(str, missing_keys[:10]))
)

print("Target lookup ready.")

**Generate UTS-style and SBTS-style temperature predictions**

In [ ]:
prediction_rows = []
sbts_model_rows = []

for backbone in BACKBONES:
    for seed in SEEDS:
        print(f"\nGenerating close-baseline predictions | {backbone} | seed={seed}")

        source_T = SOURCE_T[(backbone, seed)]

        source_bank_seed_df = source_bank_df[
            (source_bank_df["backbone"] == backbone)
            & (source_bank_df["seed"] == seed)
        ].copy()

        target_stream_seed_df = target_stream_df[
            (target_stream_df["backbone"] == backbone)
            & (target_stream_df["seed"] == seed)
        ].copy()

        assert len(source_bank_seed_df) == 50, (
            f"{backbone} seed={seed}: expected 50 source-bank rows, "
            f"got {len(source_bank_seed_df)}"
        )

        assert len(target_stream_seed_df) == 50, (
            f"{backbone} seed={seed}: expected 50 target stream rows, "
            f"got {len(target_stream_seed_df)}"
        )

        sbts_info = fit_sbts_models(
            train_df=source_bank_seed_df,
            source_T=source_T,
        )

        sbts_model_rows.append({
            "backbone": backbone,
            "seed": int(seed),
            "source_T": float(source_T),
            "entropy_train_median": float(sbts_info["entropy_train_median"]),
            "T_min_empirical": float(sbts_info["T_min_empirical"]),
            "T_max_empirical": float(sbts_info["T_max_empirical"]),
            "n_train_conditions": int(sbts_info["n_train_conditions"]),
            "target_labels_used_for_fit": False,
            "source_bank_labels_used_for_source_bank_temperatures": True,
            "note": (
                "Source-bank labels are only used indirectly because source-bank fitted "
                "temperatures were already estimated in the main pipeline."
            ),
        })

        for _, row in target_stream_seed_df.iterrows():
            corruption = str(row["corruption"])
            severity = int(row["severity"])
            condition = str(row["condition"])

            key = (backbone, seed, corruption, severity, condition)

            assert key in target_lookup.index, f"Missing logits key: {key}"

            logits_path = get_logits_path_from_lookup(target_lookup, key)
            logits, labels = load_logits_labels_any(logits_path)

            # ------------------------------------------------------------
            # UTS-style weighted NLL
            # ------------------------------------------------------------
            uts_weighted_free = select_temperature_uts_weighted_nll(
                logits=logits,
                source_T=source_T,
                no_sharpen=False,
                temperature_grid=TEMPERATURE_GRID,
                weight_mode="one_minus_confidence",
            )

            uts_weighted_no_sharpen = select_temperature_uts_weighted_nll(
                logits=logits,
                source_T=source_T,
                no_sharpen=True,
                temperature_grid=TEMPERATURE_GRID,
                weight_mode="one_minus_confidence",
            )


            # SBTS-style severity ridge

            sbts_severity_free = predict_sbts_temperature(
                model_info=sbts_info,
                row=row,
                variant="sbts_severity_ridge",
                no_sharpen=False,
            )

            sbts_severity_no_sharpen = predict_sbts_temperature(
                model_info=sbts_info,
                row=row,
                variant="sbts_severity_ridge",
                no_sharpen=True,
            )


            # SBTS-style entropy-shift ridge

            sbts_entropy_free = predict_sbts_temperature(
                model_info=sbts_info,
                row=row,
                variant="sbts_entropy_shift_ridge",
                no_sharpen=False,
            )

            sbts_entropy_no_sharpen = predict_sbts_temperature(
                model_info=sbts_info,
                row=row,
                variant="sbts_entropy_shift_ridge",
                no_sharpen=True,
            )

            method_outputs = [
                {
                    "method": "uts_weighted_nll_unconstrained",
                    "information_regime": "unlabeled_target_stream_uts_style",
                    "out": uts_weighted_free,
                    "uses_target_logits_for_fit": True,
                    "uses_source_bank": False,
                    "uses_shift_metadata": False,
                    "uses_unlabeled_target_stream": True,
                },
                {
                    "method": "uts_weighted_nll_no_sharpen",
                    "information_regime": "unlabeled_target_stream_uts_style",
                    "out": uts_weighted_no_sharpen,
                    "uses_target_logits_for_fit": True,
                    "uses_source_bank": False,
                    "uses_shift_metadata": False,
                    "uses_unlabeled_target_stream": True,
                },
                {
                    "method": "sbts_severity_ridge",
                    "information_regime": "source_bank_shift_intensity_mapping",
                    "out": sbts_severity_free,
                    "uses_target_logits_for_fit": False,
                    "uses_source_bank": True,
                    "uses_shift_metadata": True,
                    "uses_unlabeled_target_stream": False,
                },
                {
                    "method": "sbts_severity_ridge_no_sharpen",
                    "information_regime": "source_bank_shift_intensity_mapping",
                    "out": sbts_severity_no_sharpen,
                    "uses_target_logits_for_fit": False,
                    "uses_source_bank": True,
                    "uses_shift_metadata": True,
                    "uses_unlabeled_target_stream": False,
                },
                {
                    "method": "sbts_entropy_shift_ridge",
                    "information_regime": "source_bank_shift_intensity_mapping",
                    "out": sbts_entropy_free,
                    "uses_target_logits_for_fit": False,
                    "uses_source_bank": True,
                    "uses_shift_metadata": False,
                    "uses_unlabeled_target_stream": True,
                },
                {
                    "method": "sbts_entropy_shift_ridge_no_sharpen",
                    "information_regime": "source_bank_shift_intensity_mapping",
                    "out": sbts_entropy_no_sharpen,
                    "uses_target_logits_for_fit": False,
                    "uses_source_bank": True,
                    "uses_shift_metadata": False,
                    "uses_unlabeled_target_stream": True,
                },
            ]

            for item in method_outputs:
                out = item["out"]

                prediction_rows.append({
                    "split_name": "main_all_source_bank_train",
                    "heldout_type": "none",
                    "heldout_value": "none",

                    "information_regime": item["information_regime"],
                    "backbone": backbone,
                    "seed": int(seed),
                    "method": item["method"],

                    "corruption": corruption,
                    "severity": severity,
                    "condition": condition,

                    "predicted_T": float(out["predicted_T"]),
                    "raw_predicted_T": float(out["raw_predicted_T"]),
                    "source_T": float(source_T),

                    "entropy_mean": float(row["entropy_mean"]),
                    "objective_value": float(out.get("objective_value", np.nan)),
                    "raw_logT": float(out.get("raw_logT", np.nan)),
                    "T_min": float(out.get("T_min", np.nan)),
                    "T_max": float(out.get("T_max", np.nan)),
                    "no_sharpen": bool(out.get("no_sharpen", False)),

                    "uses_target_labels_for_fit": False,
                    "uses_target_logits_for_fit": bool(item["uses_target_logits_for_fit"]),
                    "uses_source_bank": bool(item["uses_source_bank"]),
                    "uses_shift_metadata": bool(item["uses_shift_metadata"]),
                    "uses_unlabeled_target_stream": bool(item["uses_unlabeled_target_stream"]),

                    "logits_path": str(logits_path),
                })

baseline_predictions_df = pd.DataFrame(prediction_rows)
sbts_model_info_df = pd.DataFrame(sbts_model_rows)

expected_prediction_rows = (
    len(BACKBONES)
    * len(SEEDS)
    * len(CORRUPTIONS)
    * len(SEVERITIES)
    * 6
)

print("Expected prediction rows:", expected_prediction_rows)
print("Actual prediction rows:", len(baseline_predictions_df))

assert len(baseline_predictions_df) == expected_prediction_rows, (
    f"Expected {expected_prediction_rows}, got {len(baseline_predictions_df)}"
)

assert baseline_predictions_df["predicted_T"].gt(0).all()

prediction_path = BASELINE_TABLE_DIR / "close_baseline_temperature_predictions.csv"
sbts_model_info_path = BASELINE_TABLE_DIR / "sbts_style_model_info.csv"

baseline_predictions_df.to_csv(prediction_path, index=False)
sbts_model_info_df.to_csv(sbts_model_info_path, index=False)

print("Saved close-baseline predictions:", prediction_path)
print("Saved SBTS model info:", sbts_model_info_path)

display(baseline_predictions_df.head())
display(sbts_model_info_df)

**Evaluate Source TS and close baselines**

In [ ]:
eval_rows = []

for _, row in target_manifest.iterrows():
    backbone = row["backbone"]
    seed = int(row["seed"])
    corruption = row["corruption"]
    severity = int(row["severity"])
    condition = row["condition"]
    logits_path = Path(row["path"])

    source_T = SOURCE_T[(backbone, seed)]

    logits, labels = load_logits_labels_any(logits_path)

    scaled_logits = apply_temperature(
        logits=logits,
        temperature=source_T,
    )

    met = metrics_from_logits(scaled_logits, labels)

    for metric_name, value in met.items():
        eval_rows.append({
            "split_name": "main_all_source_bank_train",
            "heldout_type": "none",
            "heldout_value": "none",

            "information_regime": "source_only",
            "backbone": backbone,
            "seed": seed,
            "method": "source_global_ts",

            "corruption": corruption,
            "severity": severity,
            "condition": condition,

            "temperature": float(source_T),
            "metric": metric_name,
            "value": float(value),
            "logits_path": str(logits_path),
        })


for _, row in baseline_predictions_df.iterrows():
    backbone = row["backbone"]
    seed = int(row["seed"])
    corruption = row["corruption"]
    severity = int(row["severity"])
    condition = row["condition"]
    method = row["method"]
    information_regime = row["information_regime"]

    T = float(row["predicted_T"])
    logits_path = Path(row["logits_path"])

    logits, labels = load_logits_labels_any(logits_path)

    scaled_logits = apply_temperature(
        logits=logits,
        temperature=T,
    )

    met = metrics_from_logits(scaled_logits, labels)

    for metric_name, value in met.items():
        eval_rows.append({
            "split_name": "main_all_source_bank_train",
            "heldout_type": "none",
            "heldout_value": "none",

            "information_regime": information_regime,
            "backbone": backbone,
            "seed": seed,
            "method": method,

            "corruption": corruption,
            "severity": severity,
            "condition": condition,

            "temperature": float(T),
            "metric": metric_name,
            "value": float(value),
            "logits_path": str(logits_path),
        })

baseline_eval_long = pd.DataFrame(eval_rows)

expected_eval_rows = (
    # source TS
    len(BACKBONES) * len(SEEDS) * len(CORRUPTIONS) * len(SEVERITIES) * len(METRICS)
    +
    # six close baselines
    len(BACKBONES) * len(SEEDS) * len(CORRUPTIONS) * len(SEVERITIES) * 6 * len(METRICS)
)

print("Expected eval rows:", expected_eval_rows)
print("Actual eval rows:", len(baseline_eval_long))

assert len(baseline_eval_long) == expected_eval_rows, (
    f"Expected {expected_eval_rows}, got {len(baseline_eval_long)}"
)

assert baseline_eval_long["value"].notna().all()

eval_long_path = BASELINE_TABLE_DIR / "close_baseline_eval_long.csv"
baseline_eval_long.to_csv(eval_long_path, index=False)

print("Saved eval-long table:", eval_long_path)
display(baseline_eval_long.head())

**Convert evaluation to condition-wide table**

In [ ]:
baseline_condition_wide = (
    baseline_eval_long
    .pivot_table(
        index=[
            "split_name",
            "heldout_type",
            "heldout_value",
            "information_regime",
            "backbone",
            "seed",
            "method",
            "corruption",
            "severity",
            "condition",
            "temperature",
            "logits_path",
        ],
        columns="metric",
        values="value",
        aggfunc="mean",
    )
    .reset_index()
)

baseline_condition_wide.columns.name = None

missing_metric_cols = [
    metric for metric in METRICS
    if metric not in baseline_condition_wide.columns
]

assert len(missing_metric_cols) == 0, (
    f"Missing metric columns: {missing_metric_cols}"
)

assert baseline_condition_wide["ece15"].between(0, 1).all(), "ECE15 outside [0,1]"
assert baseline_condition_wide["aece15"].between(0, 1).all(), "AECE15 outside [0,1]"

expected_wide_rows = (
    len(BACKBONES)
    * len(SEEDS)
    * len(CORRUPTIONS)
    * len(SEVERITIES)
    * 7  # Source TS + six close baselines
)

print("Expected condition-wide rows:", expected_wide_rows)
print("Actual condition-wide rows:", len(baseline_condition_wide))

assert len(baseline_condition_wide) == expected_wide_rows, (
    f"Expected {expected_wide_rows}, got {len(baseline_condition_wide)}"
)

condition_wide_path = BASELINE_TABLE_DIR / "close_baseline_condition_metric_wide.csv"
baseline_condition_wide.to_csv(condition_wide_path, index=False)

print("Saved condition-wide table:", condition_wide_path)
display(baseline_condition_wide.head())

**Compute deltas vs Source TS**

In [ ]:
source_ts_condition_wide = baseline_condition_wide[
    baseline_condition_wide["method"] == "source_global_ts"
].copy()

source_ts_keep_cols = [
    "backbone",
    "seed",
    "corruption",
    "severity",
    "condition",
] + METRICS

source_ts_reference = source_ts_condition_wide[source_ts_keep_cols].rename(
    columns={metric: f"source_ts_{metric}" for metric in METRICS}
)

duplicate_source_rows = source_ts_reference.duplicated(
    subset=["backbone", "seed", "corruption", "severity", "condition"]
).sum()

assert duplicate_source_rows == 0, (
    f"Duplicate source TS reference rows: {duplicate_source_rows}"
)

baseline_delta_wide = baseline_condition_wide[
    baseline_condition_wide["method"] != "source_global_ts"
].copy()

baseline_delta_wide = baseline_delta_wide.merge(
    source_ts_reference,
    on=["backbone", "seed", "corruption", "severity", "condition"],
    how="left",
)

assert baseline_delta_wide["source_ts_nll"].notna().all(), (
    "Missing source TS reference after merge."
)

for metric in METRICS:
    baseline_delta_wide[f"delta_{metric}_vs_source_ts"] = (
        baseline_delta_wide[metric] - baseline_delta_wide[f"source_ts_{metric}"]
    )

# Accuracy must remain unchanged under positive scalar temperature scaling
assert (
    baseline_delta_wide["delta_accuracy_vs_source_ts"].abs() < 1e-10
).all(), "Temperature scaling changed accuracy."

assert baseline_delta_wide["ece15"].between(0, 1).all()
assert baseline_delta_wide["aece15"].between(0, 1).all()
assert baseline_delta_wide["source_ts_ece15"].between(0, 1).all()
assert baseline_delta_wide["source_ts_aece15"].between(0, 1).all()

expected_delta_rows = (
    len(BACKBONES)
    * len(SEEDS)
    * len(CORRUPTIONS)
    * len(SEVERITIES)
    * 6
)

print("Expected delta rows:", expected_delta_rows)
print("Actual delta rows:", len(baseline_delta_wide))

assert len(baseline_delta_wide) == expected_delta_rows, (
    f"Expected {expected_delta_rows}, got {len(baseline_delta_wide)}"
)

baseline_delta_wide_path = BASELINE_TABLE_DIR / "close_baseline_delta_vs_source_ts_wide.csv"
baseline_delta_wide.to_csv(baseline_delta_wide_path, index=False)

print("Saved delta-wide table:", baseline_delta_wide_path)
display(baseline_delta_wide.head())

**Main close-baseline summary**

In [ ]:
baseline_delta_summary = (
    baseline_delta_wide
    .groupby(
        [
            "split_name",
            "information_regime",
            "backbone",
            "method",
        ],
        as_index=False,
    )
    .agg(
        delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
        delta_brier_mean=("delta_brier_vs_source_ts", "mean"),
        delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
        delta_aece15_mean=("delta_aece15_vs_source_ts", "mean"),
        delta_signed_gap_mean=("delta_signed_gap_vs_source_ts", "mean"),
        delta_abs_signed_gap_mean=("delta_abs_signed_gap_vs_source_ts", "mean"),

        nll_mean=("nll", "mean"),
        brier_mean=("brier", "mean"),
        ece15_mean=("ece15", "mean"),
        aece15_mean=("aece15", "mean"),
        signed_gap_mean=("signed_gap", "mean"),
        abs_signed_gap_mean=("abs_signed_gap", "mean"),
        accuracy_mean=("accuracy", "mean"),
        mean_confidence_mean=("mean_confidence", "mean"),

        source_ts_nll_mean=("source_ts_nll", "mean"),
        source_ts_brier_mean=("source_ts_brier", "mean"),
        source_ts_ece15_mean=("source_ts_ece15", "mean"),
        source_ts_aece15_mean=("source_ts_aece15", "mean"),
        source_ts_signed_gap_mean=("source_ts_signed_gap", "mean"),
        source_ts_abs_signed_gap_mean=("source_ts_abs_signed_gap", "mean"),
        source_ts_accuracy_mean=("source_ts_accuracy", "mean"),
        source_ts_mean_confidence_mean=("source_ts_mean_confidence", "mean"),

        temperature_mean=("temperature", "mean"),
        temperature_min=("temperature", "min"),
        temperature_max=("temperature", "max"),

        n_conditions=("condition", "count"),
    )
)

assert baseline_delta_summary["ece15_mean"].between(0, 1).all()
assert baseline_delta_summary["aece15_mean"].between(0, 1).all()
assert baseline_delta_summary["source_ts_ece15_mean"].between(0, 1).all()
assert baseline_delta_summary["source_ts_aece15_mean"].between(0, 1).all()

summary_path = BASELINE_TABLE_DIR / "close_baseline_delta_summary.csv"
baseline_delta_summary.to_csv(summary_path, index=False)

print("Saved close-baseline summary:", summary_path)

display(
    baseline_delta_summary
    .sort_values(["backbone", "delta_nll_mean"])
    .round(6)
)

**Summaries by severity and corruption**

In [ ]:
baseline_delta_by_severity = (
    baseline_delta_wide
    .groupby(
        [
            "backbone",
            "method",
            "severity",
        ],
        as_index=False,
    )
    .agg(
        delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
        delta_brier_mean=("delta_brier_vs_source_ts", "mean"),
        delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
        delta_aece15_mean=("delta_aece15_vs_source_ts", "mean"),
        delta_abs_signed_gap_mean=("delta_abs_signed_gap_vs_source_ts", "mean"),
        temperature_mean=("temperature", "mean"),
        temperature_min=("temperature", "min"),
        temperature_max=("temperature", "max"),
        n_conditions=("condition", "count"),
    )
)

baseline_delta_by_corruption = (
    baseline_delta_wide
    .groupby(
        [
            "backbone",
            "method",
            "corruption",
        ],
        as_index=False,
    )
    .agg(
        delta_nll_mean=("delta_nll_vs_source_ts", "mean"),
        delta_brier_mean=("delta_brier_vs_source_ts", "mean"),
        delta_ece15_mean=("delta_ece15_vs_source_ts", "mean"),
        delta_aece15_mean=("delta_aece15_vs_source_ts", "mean"),
        delta_abs_signed_gap_mean=("delta_abs_signed_gap_vs_source_ts", "mean"),
        temperature_mean=("temperature", "mean"),
        temperature_min=("temperature", "min"),
        temperature_max=("temperature", "max"),
        n_conditions=("condition", "count"),
    )
)

by_severity_path = BASELINE_TABLE_DIR / "close_baseline_delta_by_severity.csv"
by_corruption_path = BASELINE_TABLE_DIR / "close_baseline_delta_by_corruption.csv"

baseline_delta_by_severity.to_csv(by_severity_path, index=False)
baseline_delta_by_corruption.to_csv(by_corruption_path, index=False)

print("Saved severity summary:", by_severity_path)
print("Saved corruption summary:", by_corruption_path)

print("\nBy severity:")
display(
    baseline_delta_by_severity
    .sort_values(["backbone", "method", "severity"])
    .round(6)
)

print("\nBy corruption:")
display(
    baseline_delta_by_corruption
    .sort_values(["backbone", "method", "delta_nll_mean"])
    .round(6)
)

**Bootstrap confidence intervals**

In [ ]:
def bootstrap_ci_table(
    df: pd.DataFrame,
    group_cols,
    value_cols,
    n_boot: int = 1000,
    seed: int = 123,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []

    for group_key, group_df in df.groupby(group_cols):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        group_df = group_df.reset_index(drop=True)

        for value_col in value_cols:
            values = group_df[value_col].to_numpy(dtype=float)
            values = values[np.isfinite(values)]

            if len(values) == 0:
                mean_value = np.nan
                ci_low = np.nan
                ci_high = np.nan
            else:
                boot_means = []

                for _ in range(n_boot):
                    sample_idx = rng.integers(
                        low=0,
                        high=len(values),
                        size=len(values),
                    )
                    boot_means.append(float(np.mean(values[sample_idx])))

                mean_value = float(np.mean(values))
                ci_low = float(np.percentile(boot_means, 2.5))
                ci_high = float(np.percentile(boot_means, 97.5))

            row = {
                group_cols[i]: group_key[i]
                for i in range(len(group_cols))
            }

            row.update({
                "quantity": value_col,
                "mean": mean_value,
                "ci95_low": ci_low,
                "ci95_high": ci_high,
                "n_rows": int(len(group_df)),
                "n_boot": int(n_boot),
                "bootstrap_seed": int(seed),
            })

            rows.append(row)

    return pd.DataFrame(rows)


ci_value_cols = [
    "delta_nll_vs_source_ts",
    "delta_brier_vs_source_ts",
    "delta_ece15_vs_source_ts",
    "delta_aece15_vs_source_ts",
    "delta_abs_signed_gap_vs_source_ts",
]

baseline_bootstrap_ci = bootstrap_ci_table(
    df=baseline_delta_wide,
    group_cols=["backbone", "method"],
    value_cols=ci_value_cols,
    n_boot=1000,
    seed=123,
)

bootstrap_ci_path = BASELINE_TABLE_DIR / "close_baseline_delta_bootstrap_ci.csv"
baseline_bootstrap_ci.to_csv(bootstrap_ci_path, index=False)

baseline_bootstrap_ci_wide = (
    baseline_bootstrap_ci
    .pivot_table(
        index=["backbone", "method", "n_rows", "n_boot", "bootstrap_seed"],
        columns="quantity",
        values=["mean", "ci95_low", "ci95_high"],
        aggfunc="first",
    )
)

baseline_bootstrap_ci_wide.columns = [
    f"{stat}_{quantity}"
    for stat, quantity in baseline_bootstrap_ci_wide.columns
]

baseline_bootstrap_ci_wide = baseline_bootstrap_ci_wide.reset_index()

bootstrap_ci_wide_path = BASELINE_TABLE_DIR / "close_baseline_delta_bootstrap_ci_wide.csv"
baseline_bootstrap_ci_wide.to_csv(bootstrap_ci_wide_path, index=False)

print("Saved bootstrap CI table:", bootstrap_ci_path)
print("Saved bootstrap CI wide table:", bootstrap_ci_wide_path)

display(
    baseline_bootstrap_ci_wide
    .sort_values(["backbone", "mean_delta_nll_vs_source_ts"])
    .round(6)
)

**Load Safe V2 and Frozen V3 summaries**

In [ ]:
def read_csv_if_exists(path: Path):
    path = Path(path)

    if not path.exists():
        print("Missing:", path)
        return None

    df = pd.read_csv(path)
    df = normalize_backbone_column(df)

    return df


def standardize_method_summary(
    df: pd.DataFrame,
    source_block: str,
    default_method: str = None,
) -> pd.DataFrame:
    df = df.copy()
    df = normalize_backbone_column(df)

    if "method" not in df.columns:
        if "final_block" in df.columns:
            df["method"] = df["final_block"]
        elif default_method is not None:
            df["method"] = default_method
        else:
            raise ValueError(
                f"Could not infer method column for {source_block}. "
                f"Columns: {list(df.columns)}"
            )

    if default_method is not None:
        df["method"] = default_method

    if "split_name" not in df.columns:
        df["split_name"] = "main_all_source_bank_train"

    if "information_regime" not in df.columns:
        df["information_regime"] = source_block

    df["source_block"] = source_block

    wanted_cols = [
        "source_block",
        "split_name",
        "information_regime",
        "backbone",
        "method",

        "delta_nll_mean",
        "delta_brier_mean",
        "delta_ece15_mean",
        "delta_aece15_mean",
        "delta_signed_gap_mean",
        "delta_abs_signed_gap_mean",

        "nll_mean",
        "brier_mean",
        "ece15_mean",
        "aece15_mean",
        "signed_gap_mean",
        "abs_signed_gap_mean",
        "accuracy_mean",
        "mean_confidence_mean",

        "source_ts_nll_mean",
        "source_ts_brier_mean",
        "source_ts_ece15_mean",
        "source_ts_aece15_mean",
        "source_ts_signed_gap_mean",
        "source_ts_abs_signed_gap_mean",
        "source_ts_accuracy_mean",
        "source_ts_mean_confidence_mean",

        "temperature_mean",
        "temperature_min",
        "temperature_max",
        "predicted_T_mean",
        "n_conditions",
    ]

    for col in wanted_cols:
        if col not in df.columns:
            df[col] = np.nan

    return df[wanted_cols].copy()


summary_parts = []

# Close baselines from this notebook
close_baseline_summary_std = standardize_method_summary(
    df=baseline_delta_summary,
    source_block="close_baselines",
)

summary_parts.append(close_baseline_summary_std)


# ResNet Safe V2 / Frozen V3
resnet_safe_v2_candidates = [
    RESNET_ROOT / "safe_v2_binned_isotonic" / "safe_v2_main_delta_summary.csv",
    RESNET_ROOT / "stream_methods" / "safe_v2_main_delta_summary.csv",
]

resnet_v3_candidates = [
    RESNET_ROOT / "safe_v3_frozen" / "frozen_v3_main_delta_summary.csv",
    RESNET_ROOT / "stream_methods" / "frozen_v3_main_delta_summary.csv",
]

resnet_safe_v2_path = None
for p in resnet_safe_v2_candidates:
    if p.exists():
        resnet_safe_v2_path = p
        break

resnet_v3_path = None
for p in resnet_v3_candidates:
    if p.exists():
        resnet_v3_path = p
        break

resnet_safe_v2_summary = (
    read_csv_if_exists(resnet_safe_v2_path)
    if resnet_safe_v2_path is not None
    else None
)

resnet_v3_summary = (
    read_csv_if_exists(resnet_v3_path)
    if resnet_v3_path is not None
    else None
)

if resnet_safe_v2_summary is not None:
    summary_parts.append(
        standardize_method_summary(
            df=resnet_safe_v2_summary,
            source_block="resnet_safe_v2",
            default_method="safe_v2",
        )
    )

if resnet_v3_summary is not None:
    summary_parts.append(
        standardize_method_summary(
            df=resnet_v3_summary,
            source_block="resnet_frozen_v3",
            default_method="frozen_v3",
        )
    )


# ViT Safe V2 / Frozen V3
vit_safe_v2_candidates = [
    VIT_ROOT / "safe_v2_binned_isotonic" / "safe_v2_main_delta_summary.csv",
    VIT_ROOT / "stream_methods" / "safe_v2_main_delta_summary.csv",
]

vit_v3_candidates = [
    VIT_ROOT / "safe_v3_frozen" / "frozen_v3_main_delta_summary.csv",
    VIT_ROOT / "stream_methods" / "frozen_v3_main_delta_summary.csv",
]

vit_safe_v2_path = None
for p in vit_safe_v2_candidates:
    if p.exists():
        vit_safe_v2_path = p
        break

vit_v3_path = None
for p in vit_v3_candidates:
    if p.exists():
        vit_v3_path = p
        break

vit_safe_v2_summary = (
    read_csv_if_exists(vit_safe_v2_path)
    if vit_safe_v2_path is not None
    else None
)

vit_v3_summary = (
    read_csv_if_exists(vit_v3_path)
    if vit_v3_path is not None
    else None
)

if vit_safe_v2_summary is not None:
    summary_parts.append(
        standardize_method_summary(
            df=vit_safe_v2_summary,
            source_block="vit_safe_v2",
            default_method="safe_v2",
        )
    )

if vit_v3_summary is not None:
    summary_parts.append(
        standardize_method_summary(
            df=vit_v3_summary,
            source_block="vit_frozen_v3",
            default_method="frozen_v3",
        )
    )


print("Number of summary blocks loaded:", len(summary_parts))

for i, part in enumerate(summary_parts):
    print(i, part["source_block"].unique(), part.shape)

**Combined comparison table**

In [ ]:
combined_comparison = pd.concat(
    summary_parts,
    ignore_index=True,
    sort=False,
)

combined_comparison = combined_comparison[
    combined_comparison["backbone"].isin(BACKBONES)
].copy()

# Lower delta is better for NLL, Brier, ECE, AECE, and abs gap.
ranking_cols = {
    "delta_nll_mean": "rank_by_delta_nll_within_backbone",
    "delta_brier_mean": "rank_by_delta_brier_within_backbone",
    "delta_ece15_mean": "rank_by_delta_ece15_within_backbone",
    "delta_aece15_mean": "rank_by_delta_aece15_within_backbone",
    "delta_abs_signed_gap_mean": "rank_by_delta_abs_gap_within_backbone",
}

for metric_col, rank_col in ranking_cols.items():
    combined_comparison[rank_col] = (
        combined_comparison
        .groupby("backbone")[metric_col]
        .rank(method="min", ascending=True)
    )

combined_comparison_path = (
    BASELINE_TABLE_DIR / "close_baseline_vs_safev2_frozenv3_combined_comparison.csv"
)

combined_comparison.to_csv(combined_comparison_path, index=False)

print("Saved combined comparison:", combined_comparison_path)

display(
    combined_comparison[
        [
            "backbone",
            "method",
            "information_regime",
            "delta_nll_mean",
            "delta_brier_mean",
            "delta_ece15_mean",
            "delta_aece15_mean",
            "delta_abs_signed_gap_mean",
            "nll_mean",
            "ece15_mean",
            "aece15_mean",
            "temperature_mean",
            "rank_by_delta_nll_within_backbone",
            "rank_by_delta_ece15_within_backbone",
            "n_conditions",
        ]
    ]
    .sort_values(["backbone", "rank_by_delta_nll_within_backbone"])
    .round(6)
)

**Compact supervisor-facing comparison table**

In [ ]:
compact_comparison_cols = [
    "backbone",
    "method",
    "information_regime",

    "delta_nll_mean",
    "delta_brier_mean",
    "delta_ece15_mean",
    "delta_aece15_mean",
    "delta_abs_signed_gap_mean",

    "nll_mean",
    "brier_mean",
    "ece15_mean",
    "aece15_mean",
    "abs_signed_gap_mean",

    "temperature_mean",
    "temperature_min",
    "temperature_max",

    "rank_by_delta_nll_within_backbone",
    "rank_by_delta_brier_within_backbone",
    "rank_by_delta_ece15_within_backbone",
    "rank_by_delta_aece15_within_backbone",
    "rank_by_delta_abs_gap_within_backbone",

    "n_conditions",
    "source_block",
]

compact_comparison = combined_comparison[compact_comparison_cols].copy()

compact_comparison_path = (
    BASELINE_TABLE_DIR / "close_baseline_vs_safev2_frozenv3_compact_comparison.csv"
)

compact_comparison.to_csv(compact_comparison_path, index=False)

print("Saved compact supervisor comparison:", compact_comparison_path)

display(
    compact_comparison
    .sort_values(["backbone", "rank_by_delta_nll_within_backbone"])
    .round(6)
)

**Best method by metric and backbone**

In [ ]:
best_metric_cols = {
    "delta_nll_mean": "best_by_nll_delta",
    "delta_brier_mean": "best_by_brier_delta",
    "delta_ece15_mean": "best_by_ece15_delta",
    "delta_aece15_mean": "best_by_aece15_delta",
    "delta_abs_signed_gap_mean": "best_by_abs_gap_delta",
}

best_rows = []

for backbone, group_df in compact_comparison.groupby("backbone"):
    for metric_col, selection_name in best_metric_cols.items():
        valid_df = group_df[group_df[metric_col].notna()].copy()

        if len(valid_df) == 0:
            continue

        best_row = valid_df.sort_values(metric_col, ascending=True).iloc[0]

        best_rows.append({
            "backbone": backbone,
            "selection_metric": metric_col,
            "selection_name": selection_name,
            "best_method": best_row["method"],
            "best_value": float(best_row[metric_col]),
            "n_methods_compared": int(len(valid_df)),
            "interpretation": (
                "Lower is better. Values are mean deltas vs Source TS. "
                "Negative values indicate improvement over Source TS."
            ),
        })

best_method_by_metric = pd.DataFrame(best_rows)

best_method_path = BASELINE_TABLE_DIR / "close_baseline_best_method_by_metric.csv"
best_method_by_metric.to_csv(best_method_path, index=False)

print("Saved best-method table:", best_method_path)

display(best_method_by_metric)

**Copy outputs into supervisor deliverable folders**

In [ ]:
copy_targets = []

resnet_combined_tables = (
    RESNET_EXPORT_ROOT
    / "combined_resnet18_resnet50"
    / "tables"
)

if resnet_combined_tables.exists():
    copy_targets.append(resnet_combined_tables)

for backbone in ["resnet18", "resnet50"]:
    p = RESNET_EXPORT_ROOT / backbone / "tables"
    if p.exists():
        copy_targets.append(p)

vit_tables = VIT_EXPORT_ROOT / "tables"

if vit_tables.exists():
    copy_targets.append(vit_tables)

files_to_copy = [
    feasibility_path,
    feasibility_compact_path,

    target_manifest_path,
    source_bank_combined_path,
    target_stream_combined_path,
    source_ts_combined_path,
    source_entropy_ref_path,

    prediction_path,
    sbts_model_info_path,

    eval_long_path,
    condition_wide_path,
    baseline_delta_wide_path,
    summary_path,
    by_severity_path,
    by_corruption_path,

    bootstrap_ci_path,
    bootstrap_ci_wide_path,

    combined_comparison_path,
    compact_comparison_path,
    best_method_path,
]

copy_rows = []

for target_dir in copy_targets:
    target_dir.mkdir(parents=True, exist_ok=True)

    for src in files_to_copy:
        src = Path(src)

        if not src.exists():
            print("Skipping missing file:", src)
            continue

        dst = target_dir / f"19_{src.name}"
        shutil.copy2(src, dst)

        copy_rows.append({
            "target_dir": str(target_dir),
            "source_path": str(src),
            "copied_path": str(dst),
            "exists": bool(dst.exists()),
            "size_bytes": int(dst.stat().st_size) if dst.exists() else 0,
        })

copy_manifest_df = pd.DataFrame(copy_rows)

copy_manifest_path = (
    BASELINE_MANIFEST_DIR / "close_baseline_copy_manifest.csv"
)

copy_manifest_df.to_csv(copy_manifest_path, index=False)

print("Copied outputs to", len(copy_targets), "deliverable table folders.")
print("Saved copy manifest:", copy_manifest_path)

display(copy_manifest_df.head())

**Final artifact manifest and completion note**

In [ ]:
artifact_rows = []

for p in sorted(BASELINE_TABLE_DIR.glob("*.csv")):
    artifact_rows.append({
        "artifact_group": "table",
        "artifact_name": p.name,
        "path": str(p),
        "size_bytes": int(p.stat().st_size),
    })

for p in sorted(BASELINE_MANIFEST_DIR.glob("*")):
    artifact_rows.append({
        "artifact_group": "manifest",
        "artifact_name": p.name,
        "path": str(p),
        "size_bytes": int(p.stat().st_size),
    })

artifact_manifest_df = pd.DataFrame(artifact_rows)

artifact_manifest_path = (
    BASELINE_MANIFEST_DIR / "close_baseline_artifact_manifest.csv"
)

artifact_manifest_df.to_csv(artifact_manifest_path, index=False)

completion_note = {
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "output_root": str(BASELINE_ROOT),
    "purpose": (
        "Close-baseline feasibility and frozen-logit implementation for UTS-style "
        "and SBTS-style calibration baselines."
    ),
    "implemented_baselines": [
        "uts_weighted_nll_unconstrained",
        "uts_weighted_nll_no_sharpen",
        "sbts_entropy_shift_ridge",
        "sbts_entropy_shift_ridge_no_sharpen",
        "sbts_severity_ridge",
        "sbts_severity_ridge_no_sharpen",
    ],
    "not_implemented": {
        "PseudoCal": (
            "Not implemented because a faithful version likely requires target "
            "images/features and new model forward passes."
        ),
        "UTDC": (
            "Not implemented because the exact method/acronym still needs verification."
        ),
    },
    "target_labels_used_for_fitting": False,
    "target_labels_used_for_evaluation": True,
    "main_outputs": {
        "feasibility_table": str(feasibility_path),
        "temperature_predictions": str(prediction_path),
        "condition_wide": str(condition_wide_path),
        "delta_wide": str(baseline_delta_wide_path),
        "summary": str(summary_path),
        "bootstrap_ci_wide": str(bootstrap_ci_wide_path),
        "combined_comparison": str(combined_comparison_path),
        "compact_comparison": str(compact_comparison_path),
        "best_method_by_metric": str(best_method_path),
        "artifact_manifest": str(artifact_manifest_path),
        "copy_manifest": str(copy_manifest_path),
    },
    "interpretation_note": (
        "All implemented close baselines are evaluated using target labels only for metrics. "
        "Temperature fitting uses cached logits/source-bank descriptors without target labels. "
        "For NLL, Brier, ECE, AECE, and absolute confidence-accuracy gap, lower delta vs Source TS is better."
    ),
}

completion_note_path = (
    BASELINE_MANIFEST_DIR / "close_baseline_completion_note.json"
)

with open(completion_note_path, "w") as f:
    json.dump(completion_note, f, indent=2)

print("DONE.")
print("Close-baseline outputs saved in:", BASELINE_ROOT)
print("Saved artifact manifest:", artifact_manifest_path)
print("Saved completion note:", completion_note_path)

print("\nCompletion note:")
print(json.dumps(completion_note, indent=2))

print("\nMain close-baseline summary:")
display(
    baseline_delta_summary
    .sort_values(["backbone", "delta_nll_mean"])
    .round(6)
)

print("\nCompact comparison against Safe V2 / Frozen V3:")
display(
    compact_comparison
    .sort_values(["backbone", "rank_by_delta_nll_within_backbone"])
    .round(6)
)